Original DeBERTa competition experiment
Historical experiment retained for provenance. Use the repository README for the maintained entry point.


In [ ]:
# -*- coding: utf-8 -*-
"""submission.csv

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/#fileId=https%3A//storage.googleapis.com/kaggle-colab-exported-notebooks/almizankhan/submission-csv.83cd02d6-bc02-4114-af0d-be6dd083504d.ipynb%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com/20251017/auto/storage/goog4_request%26X-Goog-Date%3D20251017T054651Z%26X-Goog-Expires%3D259200%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3D72d8c801034a32f376917afaf622b1c09839ef0dffe06218c65b168d823a5399636ea7369bc9dcc8451cd238fc7dc5be27edaa08bd0228a80fb515991e21052f0d4edb81178d96178f94b75decccf75d8a1f7df67bbc6c6bcf2421497c2045d8a71f78462cd3859ff690e9884a4f547db6da153cc8fe3c4c4ed2b8161cb1c8ad6a11e9ea53749c2ac4ec556ebba900f9c5e926946aa18abedfbf9fa115ae56ccd035f4961ad5d8b6a947d5b7d9e75856e4a9ce1066dd1741e83c69c8b9741c9451b5df31a26c3e9521e63cae311178eef0a7659efe8c0d8e31a8960bd51e395fd095677ff35165ba3bbef57013cf41a9e6aeb4973594c40bfc3cf9421be8ed95
"""

import kagglehub
kagglehub.login()

import kagglehub
kagglehub.competition_download('jigsaw-agile-community-rules')
kagglehub.dataset_download('jonathanchan/deberta-v3-base')

print('Data source import complete.')

import numpy as np
import pandas as pd
import os
import warnings
import random
import torch

!pip install transformers tokenizers datasets accelerate -q
!pip install "pyarrow==15.0.0"

from datasets import Dataset, Value
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

warnings.simplefilter('ignore')

MODEL_PATH = "/kaggle/input/deberta-v3-base/deberta-v3-base"
BATCH_SIZE = 16
NUM_EPOCHS = 4
LR = 2e-5
MAX_LENGTH = 256
SEED = 42


os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_PATH = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
TEST_PATH = "/kaggle/input/jigsaw-agile-community-rules/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH) if os.path.exists(TEST_PATH) else None

print("Train shape:", train_df.shape)
if test_df is not None:
    print("Test shape:", test_df.shape)





import re
import pandas as pd

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r'\[.*?\]\(.*?\)', '', text)

    text = re.sub(r'http\S+|www\S+', '', text)

    text = re.sub(r'<.*?>', '', text)

    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\*(.*?)\*', r'\1', text)
    text = re.sub(r'>\s?', '', text)

    text = re.sub(r'[^\w\s.,!?\'"]+', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

train_df['body'] = train_df['body'].astype(str).apply(clean_text)


print("✅ Text cleaning done! Example:")
print(train_df['body'].head(5))







TEXT_COL = "body"
CONTEXT_COL = "rule"
LABEL_COL = "rule_violation"

def build_input(row):
    text = str(row.get(TEXT_COL, ""))
    if CONTEXT_COL in row and pd.notna(row[CONTEXT_COL]):
        rule = str(row[CONTEXT_COL])
        return f"Rule: {rule} [SEP] Comment: {text}"
    return text

train_df["model_input"] = train_df.apply(build_input, axis=1)
if test_df is not None:
    test_df["model_input"] = test_df.apply(build_input, axis=1)

train_sub, valid_sub = train_test_split(
    train_df, test_size=0.1, stratify=train_df[LABEL_COL], random_state=SEED
)
print(f"Train split: {train_sub.shape}, Valid split: {valid_sub.shape}")

hf_train = Dataset.from_pandas(train_sub[["model_input", LABEL_COL]].rename(columns={LABEL_COL: "label"}))
hf_valid = Dataset.from_pandas(valid_sub[["model_input", LABEL_COL]].rename(columns={LABEL_COL: "label"}))

hf_train = hf_train.cast_column("label", Value('float32'))
hf_valid = hf_valid.cast_column("label", Value('float32'))

assert os.path.isdir(MODEL_PATH), f"Model directory not found at {MODEL_PATH}. Please check your input data source."

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)

def tokenize_fn(batch):
    return tokenizer(batch["model_input"], truncation=True, max_length=MAX_LENGTH)

hf_train = hf_train.map(tokenize_fn, batched=True)
hf_valid = hf_valid.map(tokenize_fn, batched=True)
hf_train = hf_train.remove_columns(["model_input"])
hf_valid = hf_valid.remove_columns(["model_input"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

class DebertaV3ForSequenceClassificationWithFloatLabels(AutoModelForSequenceClassification):
    def forward(self, *args, **kwargs):
        if "labels" in kwargs:
            kwargs["labels"] = kwargs["labels"].float()
        return super().forward(*args, **kwargs)

model = DebertaV3ForSequenceClassificationWithFloatLabels.from_pretrained(MODEL_PATH, num_labels=1)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits.flatten()))
    auc = roc_auc_score(labels, probs)
    predictions = (probs > 0.7).astype(int)
    accuracy = accuracy_score(labels, predictions)
    return {"auc": auc, "accuracy": accuracy}

os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir="./deberta_ckpt",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="auc",
    greater_is_better=True,
    fp16=True,
    gradient_accumulation_steps=2,

    save_total_limit=1,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# --- 8. Inference and Submission ---
if test_df is not None:

    # Since your ID column is 'row_id'
    ROW_ID_COL = "row_id"

    # Convert to HF dataset
    hf_test = Dataset.from_pandas(test_df[["model_input", ROW_ID_COL]])
    hf_test = hf_test.map(tokenize_fn, batched=True)
    hf_test = hf_test.remove_columns(["model_input"])

    # Predictions
    preds = trainer.predict(hf_test)
    probs = 1 / (1 + np.exp(-preds.predictions.flatten()))

    # Create submission
    submission = pd.DataFrame({
        ROW_ID_COL: test_df[ROW_ID_COL],
        "rule_violation": probs  # Predictions column required by Kaggle
    })

    submission.to_csv("submission.csv", index=False)
    print("\n✅ Submission file created successfully!")
    display(submission.head())

else:
    print("No test set found.")
